In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q "/content/drive/MyDrive/TCC-Evelyn-2026.1/projeto/dataset_minds.zip" -d "/content/dataset_minds"
print("Dataset descompactado com sucesso na memória rápida!")

In [ ]:
import os                  
import cv2                 
import seaborn as sns      
import numpy as np         
import matplotlib.pyplot as plt  
import pandas as pd        
import time                          

from sklearn.model_selection import train_test_split           
from sklearn.preprocessing import LabelEncoder                 
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support, accuracy_score

import tensorflow as tf
from tensorflow.keras.applications import ResNet50            
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import TimeDistributed, LSTM, Dense, GlobalAveragePooling2D, Dropout, Input, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam


IMG_SIZE = 128                  # redimensionamento para 128×128 pixels
MAX_FRAMES = 30                 # a LSTM recebe sempre exatamente 30 frames por vídeo 
BATCH_SIZE = 16                 # número de vídeos processados de uma vez por atualização de peso
EPOCHS = 30                     # número máximo de épocas
BASE_PATH  = '/content/dataset_minds/dataset_minds'

In [ ]:
# Ordena numericamente pelo número do frame no nome do arquivo
def extrair_numero_frame(nome_arquivo):
        nome_sem_prefixo = nome_arquivo.replace('frame_', '')
        nome_sem_extensao = nome_sem_prefixo.replace('.jpg', '')
        numero_frame = int(nome_sem_extensao)
        return numero_frame


# Função: carrega os frames de UMA pasta de vídeo
def carregar_frames_pasta(folder_path, n_frames=MAX_FRAMES, img_size=IMG_SIZE):

    frames = []                                                 # lista que vai acumular os arrays de cada frame lido
    imagens = []


    #----- lista e ordena os arquivos de imagem da pasta
    for f in os.listdir(folder_path):                           #Lista todos os arquivos (apenas arquivos, não subpastas) dentro de folder_path
        caminho = os.path.join(folder_path, f)
        if os.path.isfile(caminho):
            imagens.append(f)

    imagens.sort(key=extrair_numero_frame)
    imagens = imagens[:n_frames]                                # Limita a lista ao número máximo de frames definido 

    if len(imagens) == 0:
        return np.zeros((n_frames, img_size, img_size, 3))      # Caso a pasta esteja completamente vazia, devolve um array de zeros


    # ---- ler, redimensionar e normalizar cada frame 
    for img_file in imagens:
        img_path = os.path.join(folder_path, img_file)          # Monta o caminho completo do arquivo de imagem
        img = cv2.imread(img_path)                              # Lê a imagem do disco como array numpy 
        if img is not None:                                     # Processa apenas se a leitura funcionou (cv2.imread retorna None se falhar)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)          # Converte de BGR (padrão do OpenCV) para RGB (padrão do TensorFlow/Keras)
            img = (img / 255.0).astype(np.float32)              # Normaliza os pixels, divide por 255 para que os valores fiquem entre 0.0 e 1.0 e converte para ponto flutuante de 32 bits
            frames.append(img)                                  # adiciona o frame processado na lista
    return np.array(frames)                                     # Converte a lista de frames em um único array numpy de shape (qtd_frames, 128, 128, 3)

In [ ]:
#Verificação do ambiente: versão do TensorFlow e disponibilidade de GPU
print("TensorFlow version:", tf.__version__)
if tf.config.list_physical_devices('GPU'):
    print("GPU disponível:")
else:
    print("Usando CPU")

In [ ]:
## O Gerador lê apenas um lote (batch) de vídeos por vez, usa na GPU e descarta.
class GeradorDeVideos(tf.keras.utils.Sequence):                     # Herda de tf.keras.utils.Sequence: protocolo que o Keras usa para geradores customizados
    def __init__(self, caminhos_pastas, rotulos_inteiros, num_classes, batch_size, max_frames, img_size, shuffle=True):

        self.caminhos = caminhos_pastas                             # guarda a lista de caminhos das pastas
        self.rotulos = rotulos_inteiros                             # guarda os rótulos numéricos correspondentes
        self.num_classes = num_classes                              # total de classes (para construir o one-hot)
        self.batch_size = batch_size                                # tamanho de cada lote
        self.max_frames = max_frames                                # frames por vídeo
        self.img_size = img_size                                    # resolução dos frames
        self.shuffle = shuffle                                      # se True, embaralha ao final de cada época

        self.indices = np.arange(len(self.caminhos))                #Cria um array de índices [0, 1, 2, ..., N-1] para controlar a ordem de acesso
        self.on_epoch_end()                                         # Embaralha os índices imediatamente na criação do gerador

    def __len__(self):
        total_caminhos = len(self.caminhos)                         #conta quantas pastas de vídeo existem no gerador.
        total_batches = total_caminhos / float(self.batch_size)     #calcula quantos lotes seriam sao necessarios quando divide pelo tamanho do batch.
        quantidade_batches = np.ceil(total_batches)                 #arredonda esse valor para cima, para não perder o último lote incompleto.
        return int(quantidade_batches)                              # retorna quantos lotes (batches) existem no total  640 vídeos / 16 por lote = 40 lotes exatos

    def on_epoch_end(self):        
        if self.shuffle:                                            # Embaralha os índices para que a próxima época veja os vídeos em ordem diferente
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        lote_caminhos = []
        lote_rotulos = []
        X_batch = []                                                # vai acumular os frames de cada vídeo do lote
        
        indices_iniciais = idx * self.batch_size                    # Seleciona os índices globais que pertencem a este lote específico
        indices_finais = (idx + 1) * self.batch_size
        batch_indices = self.indices[indices_iniciais:indices_finais]

        for i in batch_indices:                                     # Recupera os caminhos e rótulos correspondentes a esses índices
            lote_caminhos.append(self.caminhos[i])
            lote_rotulos.append(self.rotulos[i])

        for folder_path in lote_caminhos:
            frames = carregar_frames_pasta(folder_path, self.max_frames, self.img_size)
            X_batch.append(frames)

        X_batch = np.array(X_batch, dtype=np.float32)               # Empilha os vídeos do lote em um único array 5D:
        y_batch = to_categorical(lote_rotulos, num_classes=self.num_classes)            # Converte rótulos inteiros para one-hot encoding - classe 2 de 20 → [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

        return X_batch, y_batch                                     # Retorna o par (entrada, saída esperada) que o Keras usa para treinar/avaliar


# -- construção das listas de caminhos e rótulos 
caminhos_totais = []                                                # vai receber o caminho completo de cada pasta de vídeo
rotulos_totais  = []                                                # vai receber o nome da classe de cada vídeo
folders = []
todos_os_itens = os.listdir(BASE_PATH)                              # Lista todos os itens dentro de BASE_PATH

for item in todos_os_itens:                                         # Filtra apenas as subpastas
    caminho_item = os.path.join(BASE_PATH, item)
    if os.path.isdir(caminho_item):
        folders.append(item)

for folder_name in folders:
    folder_path = os.path.join(BASE_PATH, folder_name)
    class_name = folder_name.rsplit('_', 1)[0]                      # extrai o nome da classe removendo o numéro
    caminhos_totais.append(folder_path)                             # guarda o caminho
    rotulos_totais.append(class_name)                               # guarda o nome da classe

print("Total de caminhos salvos prontas para o gerador:", len(caminhos_totais))

In [ ]:
# avalia o modelo treinado e gera relatórios 
def avaliacao_modelo(model, cam_teste, rot_teste, history, classes):
    gerador_eval = GeradorDeVideos(cam_teste, rot_teste, len(classes), BATCH_SIZE, MAX_FRAMES, IMG_SIZE, shuffle=False)

    # --- GRÁFICOS DE CURVAS DE APRENDIZADO 
    if history.history['accuracy']:
        plt.figure(figsize=(14, 5))  

        # curvas de acurácia
        plt.subplot(1, 2, 1)  
        plt.plot(history.history['accuracy'], label='Treino (Aprendizado)', color='blue')
        plt.plot(history.history['val_accuracy'], label='Validacao (Generalizacao)', color='orange')

        # Se a linha laranja (validação) ficar muito abaixo da azul (treino) → overfitting
        plt.title('Curvas de Acuracia (Accuracy)\nMonitoramento de Overfitting')
        plt.xlabel('Epocas (Epochs)')
        plt.ylabel('Acuracia')
        plt.legend()

        # curvas de perda (loss)
        plt.subplot(1, 2, 2)
        plt.plot(history.history['loss'], label='Treino (Perda)', color='blue')
        plt.plot(history.history['val_loss'], label='Validacao (Perda)', color='orange')

        # Loss deve diminuir ao longo das épocas; se subir na validação → overfitting
        plt.title('Curvas de Perda (Loss)\nConvergencia e estabilidade')
        plt.xlabel('Epocas (Epochs)')
        plt.ylabel('Perda')
        plt.legend()

        plt.savefig('graficos_comportamentais.png', dpi=300, bbox_inches='tight')
        plt.show()

    else:
        print('Sem historico de treino — graficos de curva ignorados.')



    # PREVISÕES DA REDE NEURAL 
    probabilidades = model.predict(gerador_eval, verbose=0)                 #percorre todos os lotes do gerador e retorna as probabilidades
    y_pred = np.argmax(probabilidades, axis=1)                              # pega o índice da maior probabilidade de cada vídeo → classe prevista
    y_true = np.array(rot_teste)                                            # Rótulos reais do conjunto de teste



    # MATRIZ DE CONFUSÃO 
    indices_classes = range(len(classes))
    matriz_confusao = confusion_matrix(y_true, y_pred, labels=indices_classes)          # confusion_matrix[i][j] = quantas vezes a classe i foi prevista como j

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        matriz_confusao,
        annot=True,                                                         # escreve o número dentro de cada célula
        fmt='d',                                                            # formato inteiro (sem decimais)
        cmap='Blues',                                                       # escala de cor azul: mais escuro = mais ocorrências
        xticklabels=classes,                                                # rótulos do eixo X = classes previstas
        yticklabels=classes                                                 # rótulos do eixo Y = classes reais
    )
    plt.xlabel('O que a IA PREVIU (Rotulos Previstos)', fontsize=12)
    plt.ylabel('O que era REAL (Rotulos Reais)', fontsize=12)
    plt.title('Matriz de Confusao (TP, TN, FP, FN)', fontsize=14)
    plt.savefig('matriz_de_confusao.png', dpi=300, bbox_inches='tight')
    plt.show()

    # MÉTRICAS GLOBAIS 
    acuracia_global = accuracy_score(y_true, y_pred)                        # Acurácia global: total de acertos / total de vídeos testados

    # Calcula precision, recall e F1 fazendo a média entre todas as classes (macro)
    precision_global, recall_global, f1_global, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

    # Precision: de tudo que a rede disse "é X", quantos realmente eram X?
    # Recall:    de todos os X reais, quantos a rede reconheceu?
    # F1-Score:  média harmônica de precision e recall (equilíbrio entre os dois)
    print('\n' + '=' * 50)
    print('======= METRICAS GLOBAIS DO MODELO (RESUMO) =======')
    print('=' * 50)
    print(f' -> Acuracia (Accuracy): {acuracia_global:.4f}')
    print(f' -> F1-Score (Media):    {f1_global:.4f}')
    print(f' -> Precisao (Precision):{precision_global:.4f}')
    print(f' -> Revocacao (Recall):  {recall_global:.4f}')
    print('=' * 50)

    # RESULTADOS DETALHADOS POR CLASSE + EXPORTAÇÃO CSV ─────────────────
    print('\nRESULTADOS DETALHADOS POR CLASSE')
    print('-' * 50)

    
    precision_arr, recall_arr, f1_arr, support_arr = precision_recall_fscore_support(   # Calcula precision, recall e F1 individualmente para cada classe
        y_true, y_pred,
        labels=list(range(len(classes))),
        average=None,
        zero_division=0
    )

    linhas_csv = []                     

    # Itera sobre cada classe com seu índice numérico e nome
    for i, nome_classe in enumerate(classes):
        saidas_previstas = []                               # lista dos nomes das classes que o modelo retornou para este sinal
        total_real = matriz_confusao[i].sum()               # Linha i da matriz de confusão = todos os vídeos reais da classe i
        verdadeiros_positivos = matriz_confusao[i][i]       # vídeos reais da classe i que foram previstos corretamente como i

        if total_real > 0:
            acuracia_classe = verdadeiros_positivos / total_real
        else:
            acuracia_classe = 0

        print(f'\nSinal Real: {nome_classe.upper()}')
        print(f'  Acuracia Isolada: {acuracia_classe:.2f}')
        print(f'  Precision: {precision_arr[i]:.2f}')
        print(f'  Recall:    {recall_arr[i]:.2f}')
        print(f'  F1-Score:  {f1_arr[i]:.2f}')

        for j, nome_classe_prevista in enumerate(classes):          # Percorre as colunas da linha i para ver para onde os vídeos foram classificados
            quantidade_prevista = matriz_confusao[i][j]
            if quantidade_prevista > 0:                             # ignora combinações que nunca ocorreram
                if i == j:
                    status = '✅ Acertou'
                else:
                    status = '❌ Confundiu com'

                print(f'  -> {status}: {nome_classe_prevista} ({quantidade_prevista} vezes)')
                saidas_previstas.append(nome_classe_prevista)

        # Monta o dicionário que representa a linha desta classe no CSV
        linhas_csv.append({
            'Classe Esperada':           nome_classe,
            'Respostas da IA':           ', '.join(saidas_previstas),  # classes que o modelo retornou
            'Acuracia':                  round(acuracia_classe, 4),
            'Precisao':                  round(precision_arr[i], 4),
            'Revocacao (Recall)':        round(recall_arr[i], 4),
            'F1-Score':                  round(f1_arr[i], 4),
            'Total de Videos (Support)': int(support_arr[i])             # total de vídeos reais desta classe
        })

    df_metricas = pd.DataFrame(linhas_csv)
    csv_path    = 'metricas_completas_por_classe.csv'
    df_metricas.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f'\n Planilha de metricas salva com sucesso em: {csv_path}')

In [ ]:
# Divisão treino/teste e instanciação dos geradores
DRIVE_PATH = '/content/drive/MyDrive/TCC-Evelyn-2026.1/projeto'
CSV_TREINO = f'{DRIVE_PATH}/dados_treino_oficial.csv'                   # salva quais vídeos são de treino
CSV_TESTE  = f'{DRIVE_PATH}/dados_teste_oficial.csv'                    # salva quais vídeos são de teste

encoder = LabelEncoder()                                                # converte nomes de classe em inteiros
y_integers = encoder.fit_transform(rotulos_totais)                      # aprende o mapeamento e já converte
class_names = encoder.classes_                                          # encoder.classes_ é um array com os nomes das classes em ordem alfabética
n_classes = len(class_names)

if os.path.exists(CSV_TREINO) and os.path.exists(CSV_TESTE):
    print("Carregando split congelado do Drive...")
    df_treino = pd.read_csv(CSV_TREINO)                                 # carrega o CSV de treino como DataFrame
    df_teste  = pd.read_csv(CSV_TESTE)                                  # carrega o CSV de teste como DataFrame
    
    cam_train = df_treino['Caminho'].tolist()  
    rot_train = df_treino['Rotulo'].tolist()   
    cam_test  = df_teste['Caminho'].tolist()   
    rot_test  = df_teste['Rotulo'].tolist()    

else:
    print("CSVs não encontrados. Criando e salvando split...")

    cam_train, cam_test, rot_train, rot_test = train_test_split(
        caminhos_totais, 
        y_integers,
        test_size = 0.2,                                                #20% vai para teste
        random_state = 20,                                              #garante que a divisão seja igual toda vez que rodar
        stratify = y_integers                                           #garante proporção igual de cada classe nos dois conjuntos
    )

    pd.DataFrame({'Caminho': cam_train, 'Rotulo': rot_train}).to_csv(CSV_TREINO, index=False)
    pd.DataFrame({'Caminho': cam_test,  'Rotulo': rot_test }).to_csv(CSV_TESTE,  index=False)
    print("Split salvo! Próximas execuções vão reusar este split.")

print(f"Classes cadastradas: {class_names}")
print(f"Vídeos no Treino: {len(cam_train)} | Vídeos no Teste: {len(cam_test)}")

gerador_treino = GeradorDeVideos(cam_train, rot_train, n_classes, BATCH_SIZE, MAX_FRAMES, IMG_SIZE)
gerador_teste  = GeradorDeVideos(cam_test,  rot_test,  n_classes, BATCH_SIZE, MAX_FRAMES, IMG_SIZE)

In [ ]:
# Definição e compilação da arquitetura da rede neural

n_classes = len(class_names)

base_model = ResNet50(
    weights='imagenet',                                 #usa pesos pré-treinados em 1 milhão de imagens (ImageNet)
    include_top=False,                                  #remove a camada de classificação original da ResNet
    pooling='avg',                                      #média dos valores espaciais de cada mapa de características da ResNet.
    input_shape=(IMG_SIZE, IMG_SIZE, 3)                 #cada frame individual
)

base_model.trainable = False                            # Congela todos os pesos da ResNet: ela NÃO aprenderá durante o nosso treino

model = Sequential([
    TimeDistributed(base_model, input_shape=(MAX_FRAMES, IMG_SIZE, IMG_SIZE, 3)),       # aplica a ResNet independentemente em cada um dos 30 frames
    TimeDistributed(Dense(512, activation='relu')),                                     # Reduz o vetor de 2048 features para 512, mantendo a dimensão temporal
    BatchNormalization(),                                                               # Normaliza os 512 valores de cada frame para média ≈ 0 e variância ≈ 1
    LSTM(256, return_sequences=False),                                                  # Recebe a sequência completa de 30 frames (cada um com 512 features)
    Dropout(0.5),                                                                       # desativa aleatoriamente 50% dos 256 neurônios a cada passo
    Dense(n_classes, activation='softmax')                                              # softmax transforma os 20 valores brutos em 20 probabilidades que somam 1.0
])


model.compile(
    # Adam: otimizador que ajusta os pesos usando gradiente descendente com momento adaptativo
    # learning_rate=0.0005 → passo de aprendizado (menor = mais estável, mas mais lento)
    # weight_decay=0.0005  → penaliza pesos muito grandes (regularização L2, evita overfitting)

    # CategoricalCrossentropy: mede o erro quando os rótulos são vetores one-hot
    # label_smoothing=0.1 → suaviza os rótulos: em vez de [0,0,1,0], usa [0.005, 0.005, 0.91, 0.005, ...]
    #                        evita que a rede fique 100% confiante, tornando-a mais robusta
    optimizer=Adam(
        learning_rate=0.0005, 
        weight_decay=0.0005),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Callbacks, treinamento e salvamento do modelo

early_stop = EarlyStopping(
    monitor='val_accuracy',                                 # observa a acurácia no conjunto de validação (teste)
    patience=15,                                            # aguarda 15 épocas sem melhora antes de parar
    min_delta=0.005,                                        # considera melhora apenas se a acurácia subir pelo menos 0.5%
    restore_best_weights=True,                              # ao parar, reverte os pesos para a época de melhor val_accuracy
    start_from_epoch=10                                     # s só começa a monitorar a partir da época 10
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',                                     # observa a perda de validação 
    factor=0.5,                                             #quando acionado, multiplica a taxa de aprendizado por 0.5
    patience=5,                                             #  espera 5 épocas sem melhora na val_loss antes de reduzir
    min_lr=1e-6,                                            # taxa mínima = 0.000001 (evita que chegue a zero e pare tudo)
    verbose=1                                               # imprime uma mensagem no console quando a taxa é reduzida
)

checkpoint = ModelCheckpoint(
    filepath='/content/drive/MyDrive/TCC-Evelyn-2026.1/projeto/backup_modelo.keras',
    save_best_only=False,
    save_freq='epoch',
    verbose=1
)

# --- Treinamento 
print("Iniciando o treinamento com os parâmetros definidos...")


history = model.fit(
    gerador_treino,
    validation_data=gerador_teste,                          # lotes de validação ao final de cada época
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr, checkpoint]
)

model.save('/content/drive/MyDrive/TCC-Evelyn-2026.1/projeto/modelo_lstm_final.keras')

avaliacao_modelo(model, cam_test, rot_test, history, class_names)